# Results Analysis – Phishing URL Detection

Reloads the trained model and inspects its behaviour on the test set.
Produces:
* Confusion matrix
* ROC & PR curves
* Feature-importance chart
* Error-case inspection (false positives / false negatives)

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve,
)

from src.utils import load_config

cfg = load_config()
model = joblib.load('../models_saved/phishing_rf_model.joblib')
feature_cols = json.loads(open('../models_saved/feature_columns.json').read())

test = pd.read_csv('../data/processed/test.csv')
X_test = test[feature_cols]
y_test = test[cfg['columns']['label_column']]

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing'], digits=4))

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Phish'], yticklabels=['Legit', 'Phish'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## ROC & PR curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
precision, recall, _ = precision_recall_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, label=f'AUC={auc(fpr, tpr):.3f}')
axes[0].plot([0,1],[0,1],'--', color='grey')
axes[0].set_title('ROC'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()

axes[1].plot(recall, precision, label=f'PR-AUC={auc(recall, precision):.3f}')
axes[1].set_title('Precision–Recall'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend()
plt.tight_layout(); plt.show()

## Top feature importances

In [ ]:
clf = model.steps[-1][1]
importances = clf.feature_importances_
imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(20)

plt.figure(figsize=(9, 6))
sns.barplot(x=imp.values, y=imp.index, orient='h', palette='viridis')
plt.title('Top 20 Feature Importances'); plt.xlabel('Importance')
plt.tight_layout(); plt.show()

## Error analysis – false positives & false negatives

In [ ]:
err = test.copy()
err['y_pred']  = y_pred
err['y_proba'] = y_proba
err['error_type'] = np.where((err['CLASS_LABEL']==0) & (err['y_pred']==1), 'FP',
                     np.where((err['CLASS_LABEL']==1) & (err['y_pred']==0), 'FN', 'correct'))

print('Counts:'); print(err['error_type'].value_counts())
err[err['error_type'] != 'correct'].sort_values('y_proba', ascending=False).head(10)

## Inference-time benchmark

In [ ]:
import time

start = time.time()
_ = model.predict(X_test)
elapsed = time.time() - start
print(f'Total time: {elapsed*1000:.1f} ms for {len(X_test)} rows')
print(f'Per-sample : {(elapsed/len(X_test))*1e6:.1f} microseconds')